In [ ]:
import numpy as np

movies = [
    "Inception",
    "Interstellar",
    "Titanic",
    "Avatar",
    "The Matrix",
    "Gladiator",
    "Joker",
    "Toy Story",
    "Avengers",
    "Frozen",
    "Shrek",
    "The Godfather"
]


ratings_display = [
    [5, 4, "?", 5, "?", 4],
    [4, 5, 2, "?", 4, 4],
    [2, "?", 5, 1, "?", 2],
    [5, 4, "?", 4, 5, "?"],
    [5, "?", 4, 5, 4, 5],
    ["?", 4, 5, "?", 3, 4],
    [4, "?", 5, 4, "?", 5],
    [3, 4, "?", 3, 5, "?"],
    [5, 5, 4, "?", 4, 5],
    ["?", 3, 2, 2, 5, 3],
    [3, "?", 4, 3, 4, "?"],
    [5, 4, 5, "?", "?", 5]
]


print("Original rating matrix:")
print()

print(
    "Movie".ljust(18),
    "User1 User2 User3 User4 User5 User6"
)

for movie, row in zip(movies, ratings_display):
    print(movie.ljust(18), row)


Y = np.array([
    [0 if value == "?" else value for value in row]
    for row in ratings_display
], dtype=float)


print("\n\nActual Y matrix:")
print(Y)


# ============================================================
# 3. CREATE R
# ============================================================

# R tells us whether a rating exists.
#
# R[i, j] = 1 -> movie i was rated by user j
# R[i, j] = 0 -> movie i was NOT rated by user j

R = (Y != 0).astype(float)

print("\nR matrix:")
print(R)


# ============================================================
# 4. CHECK ONE EXAMPLE
# ============================================================

print("\nInception / User1:")
print("Y =", Y[0, 0])
print("R =", R[0, 0])

print("\nInception / User3:")
print("Y =", Y[0, 2])
print("R =", R[0, 2])


# ============================================================
# 5. NORMALIZE RATINGS
# ============================================================

def normalize_ratings(Y, R):
    # Number of users who rated each movie
    count = np.sum(R, axis=1, keepdims=True)

    # Sum of ratings for each movie
    rating_sum = np.sum(Y * R, axis=1, keepdims=True)

    # Average rating for each movie
    movie_mean = np.divide(
        rating_sum,
        count,
        out=np.zeros_like(rating_sum),
        where=count != 0
    )

    Ynorm = (Y - movie_mean) * R
    return Ynorm, movie_mean


Ynorm, Ymean = normalize_ratings(Y, R)

print("\nY normalized:")
print(Ynorm)

print("\nMovie means:")
print(Ymean.flatten())


# ============================================================
# 6. COLLABORATIVE FILTERING COST FUNCTION
# ============================================================

def cofi_cost_func(X, W, b, Y, R, lambda_):
    predictions = X @ W.T + b
    errors = predictions - Y
    errors = errors * R

    # Main cost
    cost = 1/2 * np.sum(errors ** 2)

    # Regularization
    regularization = (
        lambda_ / 2
        * (
            np.sum(X ** 2)
            + np.sum(W ** 2)
        )
    )

    return cost + regularization


 def cofi_cost_func_V2(X, W, b, Y, R, lambda_):
    nm, nu = Y.shape
    J = 0
    for j in range(nu): 
        w = W[j, :]
        b_j = b[0, j]
        for i in range(nm): 
            x = X[i, :]
            y = Y[i, j]
            r = R[i, j] 
            J += np.square(r * (np.dot(w, x) + b_j - y))
    J = J / 2
    J += (lambda_ / 2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
    return J

# ============================================================
# 7. GRADIENT DESCENT
# ============================================================

def train_model(
    X,
    W,
    b,
    Y,
    R,
    lambda_,
    learning_rate,
    iterations
):

    for iteration in range(iterations):
        predictions = X @ W.T + b
        errors = (predictions - Y) * R

        grad_X = errors @ W + lambda_ * X
        grad_W = errors.T @ X + lambda_ * W
        grad_b = np.sum(errors, axis=0, keepdims=True)

        X = X - learning_rate * grad_X
        W = W - learning_rate * grad_W
        b = b - learning_rate * grad_b

        if iteration % 500 == 0:
            cost = cofi_cost_func(
                X,
                W,
                b,
                Y,
                R,
                lambda_
            )

            print(
                f"Iteration {iteration:5d} "
                f"Cost: {cost:.4f}"
            )

    return X, W, b


np.random.seed(42)

num_movies = Y.shape[0]
num_users = Y.shape[1]

num_features = 3
X = np.random.randn(
    num_movies,
    num_features
) * 0.1

W = np.random.randn(
    num_users,
    num_features
) * 0.1

b = np.zeros(
    (1, num_users)
)


print("\nShapes:")
print("X:", X.shape)
print("W:", W.shape)
print("b:", b.shape)


# ============================================================
# 9. TRAIN
# ============================================================

lambda_ = 0.1
learning_rate = 0.03
iterations = 5000

X, W, b = train_model(
    X,
    W,
    b,
    Ynorm,
    R,
    lambda_,
    learning_rate,
    iterations
)


# ============================================================
# 10. PREDICT RATINGS FOR ALL EXISTING USERS
# ============================================================

predictions_normalized = X @ W.T + b

# Add movie means back
predictions = predictions_normalized + Ymean

print("\nPredicted rating matrix:")
print(np.round(predictions, 2))


# ============================================================
# 11. ADD A NEW USER
# ============================================================

new_user_ratings = np.array([
    5,      # Inception
    4,      # Interstellar
    0,      # Titanic
    5,      # Avatar
    4,      # The Matrix
    0,      # Gladiator
    5,      # Joker
    0,      # Toy Story
    4,      # Avengers
    0,      # Frozen
    0,      # Shrek
    5       # The Godfather
], dtype=float)


print("\nNew user's ratings:")
for movie, rating in zip(movies, new_user_ratings):
    print(movie.ljust(18), rating)


# ============================================================
# 12. ADD NEW USER TO Y
# ============================================================

Y_new = np.c_[
    new_user_ratings,
    Y
]

R_new_user = (
    new_user_ratings != 0
).astype(float)


# Add new user's R column

R_new = np.c_[
    R_new_user,
    R
]


print("\nNew Y shape:", Y_new.shape)
print("New R shape:", R_new.shape)


# ============================================================
# 13. NORMALIZE AGAIN
# ============================================================

Ynorm_new, Ymean_new = normalize_ratings(
    Y_new,
    R_new
)


# ============================================================
# 14. TRAIN AGAIN INCLUDING THE NEW USER
# ============================================================

num_movies = Y_new.shape[0]
num_users = Y_new.shape[1]

num_features = 3

np.random.seed(42)

X_new = np.random.randn(
    num_movies,
    num_features
) * 0.1

W_new = np.random.randn(
    num_users,
    num_features
) * 0.1

b_new = np.zeros(
    (1, num_users)
)


X_new, W_new, b_new = train_model(
    X_new,
    W_new,
    b_new,
    Ynorm_new,
    R_new,
    lambda_=0.1,
    learning_rate=0.03,
    iterations=5000
)


# ============================================================
# 15. PREDICT FOR THE NEW USER
# ============================================================

all_predictions_normalized = (
    X_new @ W_new.T + b_new
)

all_predictions = (
    all_predictions_normalized
    + Ymean_new
)


# New user is column 0
new_user_predictions = all_predictions[:, 0]


print("\nPredictions for new user:")

for movie, rating in zip(
    movies,
    new_user_predictions
):
    print(
        movie.ljust(18),
        round(rating, 2)
    )


# ============================================================
# 16. FIND RECOMMENDATIONS
# ============================================================

ranking = np.argsort(
    -new_user_predictions
)


print("\nRecommended movies:")

for movie_index in ranking:
    if R_new_user[movie_index] == 1:
        continue

    print(
        movies[movie_index],
        "-> predicted rating:",
        round(new_user_predictions[movie_index], 2)
    )